In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "7"

In [2]:
import json
from pathlib import Path

import datasets as hfds
import numpy as np
import torch
import pandas as pd
from matplotlib import pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel

In [3]:
ROOT = Path("..")

In [4]:
with (ROOT / "metadata/nsd_cococlip_categories.json").open() as f:
    categories = json.load(f)
target_id_map = {coco_id: ii for ii, coco_id in enumerate(categories.values())}
target_label_map = {target_id_map[coco_id]: label for label, coco_id in categories.items()}
print(categories)

{'motorcycle': 3, 'airplane': 4, 'bus': 5, 'train': 6, 'fire hydrant': 10, 'stop sign': 11, 'horse': 17, 'sheep': 18, 'cow': 19, 'elephant': 20, 'zebra': 22, 'giraffe': 23, 'umbrella': 25, 'skis': 30, 'snowboard': 31, 'kite': 33, 'skateboard': 36, 'surfboard': 37, 'tennis racket': 38, 'pizza': 53, 'cake': 55, 'bed': 59, 'toilet': 61, 'clock': 74}


In [5]:
cluster_labels = [f"photo of {label}" for label in categories]
print("\n".join(cluster_labels[:10]))

photo of motorcycle
photo of airplane
photo of bus
photo of train
photo of fire hydrant
photo of stop sign
photo of horse
photo of sheep
photo of cow
photo of elephant


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [7]:
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14", use_fast=True)
model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
model = model.to(device)

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
recons = torch.load(ROOT / "datasets/mindeye2_subj01_shared1000_recons.pt")
image_ids = recons["image_ids"]
recons = recons["reconstructions"]
print(image_ids.shape, recons.shape)
print(image_ids.dtype, recons.dtype)
print(recons.min().item(), recons.max().item())

torch.Size([1000]) torch.Size([1000, 3, 256, 256])
torch.int64 torch.float16
0.0 1.0


In [9]:
images = [Image.fromarray((img.permute(1, 2, 0) * 255).to(torch.uint8).numpy()) for img in recons]

In [10]:
logits = []

batch_size = 100
with torch.inference_mode():
    for ii in tqdm(range(10)):
        batch = [images[idx] for idx in range(ii * batch_size, (ii + 1) * batch_size)]
        inputs = processor(text=cluster_labels, images=batch, return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = model(**inputs)
        logit = outputs["logits_per_image"].cpu().numpy()
        logits.append(logit)
logits = np.concatenate(logits)

100%|██████████| 10/10 [00:05<00:00,  1.90it/s]


In [11]:
meta_df = pd.read_csv(ROOT / "metadata/nsd_cococlip_metadata.csv")

sub_df = meta_df.query("sub == 'subj01' and split == 'shared1000'")
sub_df["target"] = sub_df["category_id"].map(target_id_map)

print(sub_df.shape)
sub_df.head()

(559, 9)


,sub,ses,run,trial_id,onset,nsd_id,split,category_id,target
7,subj01,1,1,35,164.0,44980,shared1000,4,1
12,subj01,1,1,55,252.0,53052,shared1000,74,23
52,subj01,1,4,189,16.0,44980,shared1000,4,1
68,subj01,1,4,232,204.0,5602,shared1000,11,5
70,subj01,1,4,245,260.0,21601,shared1000,25,12


In [12]:
pred_df = pd.DataFrame({"nsd_id": image_ids.numpy(), "pred": logits.argmax(axis=1)})
pred_df = pred_df.merge(sub_df.loc[:, ["nsd_id", "target"]], how="inner", on="nsd_id")
pred_df["correct"] = pred_df["pred"] == pred_df["target"]
print(pred_df.shape)
acc = pred_df["correct"].mean()
print(f"{100 * acc:.1f}%")

(559, 4)
78.0%


In [13]:
dataset = hfds.load_dataset("clane9/NSD-Flat", download_config=hfds.DownloadConfig(num_proc=8))
dataset = hfds.concatenate_datasets([dataset["train"], dataset["test"]])

dataset = dataset.filter(
    function=lambda subid, keep: subid == 0 and keep,
    input_columns=["subject_id", "shared1000"],
).select_columns(["nsd_id", "image"])
print(dataset)

Resolving data files:   0%|          | 0/54 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/44 [00:00<?, ?it/s]

Dataset({
    features: ['nsd_id', 'image'],
    num_rows: 3000
})


In [14]:
shared1000_ids = set(image_ids.numpy().tolist())
stim_images = {sample["nsd_id"]: sample["image"] for sample in tqdm(dataset)}

100%|██████████| 3000/3000 [00:06<00:00, 474.96it/s]


In [15]:
recon_images = {int(nsd_id): img for nsd_id, img in zip(image_ids, images)}

In [16]:
def imstack(img1, img2):
    img = Image.new("RGB", (img1.width + img2.width, img1.height))
    img.paste(img1, (0, 0))
    img.paste(img2, (img1.width, 0))
    return img

In [17]:
def plot_image_recon(nsd_id, target, pred):
    correct = target == pred
    img = imstack(stim_images[nsd_id], recon_images[nsd_id])
    w, h = img.size
    box = Rectangle((0, 0), w, h, color="none", ec="g" if correct else "r", lw=4.0)
    plt.imshow(img, interpolation="none")
    title = f"{target_label_map[target]} -> {target_label_map[pred]}"
    plt.title(title, fontsize="small", pad=2)
    ax = plt.gca()
    ax.add_artist(box)
    plt.axis("off")

In [18]:
ploth = 2.0
plotw = 2 * ploth * 0.95
nr = 8
nc = 8
f, axs = plt.subplots(nr, nc, figsize=(plotw * nc, ploth * nr), squeeze=False)
axs = axs.flatten()

example_df = pred_df.drop_duplicates("nsd_id").sort_values(["correct", "nsd_id"], ascending=False)
example_ids = np.linspace(0, len(example_df) - 1, nr * nc, dtype=np.int64)

for ii, ax in enumerate(axs):
    plt.sca(ax)
    row = example_df.iloc[example_ids[ii]]
    plot_image_recon(row["nsd_id"], row["target"], row["pred"])

plt.tight_layout(pad=0.5)
plt.savefig("mindeyev2_predictions.png")
plt.close()